In [72]:
# necessary imports

import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

## Initial Data Cleaning and Preprocessing

In [73]:
# read in csv

merged = pd.read_csv("../data/merged/merged_arrests_acs_2023.csv")
merged.head()

,ZCTA,total_arrests,felonies,misdemeanors,violations,AMERICAN INDIAN/ALASKAN NATIVE,ASIAN / PACIFIC ISLANDER,BLACK,BLACK HISPANIC,UNKNOWN,...,pct_native,pct_asian,pct_hispanic,arrest_rate_per_1000,HISPANIC_ARRESTS,black_arrest_rate_per_1000,white_arrest_rate_per_1000,asian_pacific_islander_arrest_rate_per_1000,american_indian/alaskan_native_arrest_rate_per_1000,hispanic_arrest_rate_per_1000
0,83,172,75,97,0,4,3,65,20,6,...,NaN,NaN,NaN,NaN,63,NaN,NaN,NaN,NaN,NaN
1,10001,4041,1660,2291,8,14,116,1798,371,106,...,0.000000,0.179786,0.190756,138.966264,1515,615.121451,34.596723,22.188217,NaN,273.120606
2,10002,2107,947,1137,8,1,166,924,209,24,...,0.000914,0.365441,0.246911,27.901002,780,140.106141,10.655408,6.015147,14.492754,41.832028
3,10003,1742,856,877,8,4,60,822,156,32,...,0.001263,0.178913,0.098189,32.364143,559,348.452734,7.685615,6.230530,58.823529,105.771050
4,10004,61,17,44,0,0,5,31,2,2,...,0.000000,0.238194,0.048774,15.741935,13,116.104869,4.187605,5.417118,NaN,68.783069


In [74]:
# general info and summary stats

print(merged.info())
print(merged.describe(include="all").transpose().head(20))

# count missing by column

missing = merged.isna().sum().sort_values(ascending=False)
missing[missing > 0]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 32 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   ZCTA                                                 190 non-null    int64  
 1   total_arrests                                        190 non-null    int64  
 2   felonies                                             190 non-null    int64  
 3   misdemeanors                                         190 non-null    int64  
 4   violations                                           190 non-null    int64  
 5   AMERICAN INDIAN/ALASKAN NATIVE                       190 non-null    int64  
 6   ASIAN / PACIFIC ISLANDER                             190 non-null    int64  
 7   BLACK                                                190 non-null    int64  
 8   BLACK HISPANIC                                       190 non-null    i

american_indian/alaskan_native_arrest_rate_per_1000    50
black_arrest_rate_per_1000                             11
hispanic_arrest_rate_per_1000                           9
asian_pacific_islander_arrest_rate_per_1000             9
white_arrest_rate_per_1000                              9
pct_hispanic                                            9
pct_asian                                               9
pct_native                                              9
pct_black                                               9
pct_white                                               9
native_count                                            4
arrest_rate_per_1000                                    4
below_poverty                                           4
median_income                                           4
hispanic_count                                          4
asian_count                                             4
black_count                                             4
white_count   

In [75]:
# drop rows where total_pop is missing or 0

df = merged.dropna(subset=["total_pop"]).copy()
df = df[df["total_pop"] > 0]

In [76]:
# ensure cols to numeric 

numeric_cols = [
    "total_arrests", "felonies", "misdemeanors", "violations",
    "total_pop", "white_count", "black_count", "native_count",
    "asian_count", "hispanic_count", "median_income", "below_poverty",
    "pct_white", "pct_black", "pct_native", "pct_asian", "pct_hispanic",
    "arrest_rate_per_1000"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [77]:
# drop insane values

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["arrest_rate_per_1000"])

In [78]:
# remove extreme outliers

upper_limit = df["arrest_rate_per_1000"].mean() + 3 * df["arrest_rate_per_1000"].std()
df = df[df["arrest_rate_per_1000"] < upper_limit]

In [79]:
# check key cols
 
key_cols = ["median_income", "pct_black", "pct_hispanic", "pct_asian", "pct_white"]
print(df[key_cols].isna().sum())

print(f"Final ZIPs retained: {df.shape[0]}")
df.describe().T.loc[["total_arrests", "total_pop", "arrest_rate_per_1000"]]

median_income    0
pct_black        0
pct_hispanic     0
pct_asian        0
pct_white        0
dtype: int64
Final ZIPs retained: 178


,count,mean,std,min,25%,50%,75%,max
total_arrests,178.0,1098.095506,951.950627,1.000000,329.250000,831.000000,1684.750000,5360.000000
total_pop,178.0,47699.747191,26151.578964,2195.000000,28003.500000,42456.500000,68430.750000,107060.000000
arrest_rate_per_1000,178.0,22.514149,17.818314,0.242542,9.727657,17.510165,31.969244,90.590112


In [81]:
df = df[df["total_pop"] > 20000].copy()
print(f"ZIPs kept: {df.shape[0]}")

ZIPs kept: 151


## Exploratory and Analytical Section

### 1. Descriptive Grounding (What does NYC look like in this data?)

#### 1.1 City Wide Aggregrates

In [82]:
# citywide totals from merged df

tot_arrests = df["total_arrests"].sum()
tot_pop = df["total_pop"].sum()
overall_rate = tot_arrests / tot_pop * 1000

tot_black_arrests = df.get("BLACK", pd.Series(0)).sum()
tot_white_arrests = df.get("WHITE", pd.Series(0)).sum()
tot_hisp_arrests = (df.get("WHITE HISPANIC", 0) + df.get("BLACK HISPANIC", 0)).sum()

tot_black_pop = df["black_count"].sum()
tot_white_pop = df["white_count"].sum()
tot_hisp_pop = df["hispanic_count"].sum()

print("Total arrests (2023, NYPD, in-scope ZIPs):", int(tot_arrests))
print("Total population (ACS 2023 5-year, these ZIPs):", int(tot_pop))
print("Overall arrest rate per 1,000:", overall_rate)

print("Black arrests:", int(tot_black_arrests), " | Black pop:", int(tot_black_pop))
print("White arrests:", int(tot_white_arrests), " | White pop:", int(tot_white_pop))
print("Hispanic arrests:", int(tot_hisp_arrests), " | Hispanic pop:", int(tot_hisp_pop))

Total arrests (2023, NYPD, in-scope ZIPs): 189260
Total population (ACS 2023 5-year, these ZIPs): 8179220
Overall arrest rate per 1,000: 23.13912573570585
Black arrests: 87590  | Black pop: 1721635
White arrests: 18601  | White pop: 2520787
Hispanic arrests: 69365  | Hispanic pop: 2356066


Note: These numbers can ground the foundation for the analysis.

#### 1.2 Distribution of arrest intensity

In [83]:
df["arrest_rate_per_1000"].describe()

count    151.000000
mean      23.266567
std       16.861986
min        0.614019
25%       10.517272
50%       18.608023
75%       32.908930
max       90.590112
Name: arrest_rate_per_1000, dtype: float64

#### 1.3 Top and bottom zip codes according to arrest_rate_per_1000

In [90]:
top10 = df.sort_values("arrest_rate_per_1000", ascending=False)[
    ["ZCTA", "total_arrests", "total_pop", "arrest_rate_per_1000",
     "pct_black", "pct_hispanic", "pct_white", "pct_asian", "median_income"]
].head(10)

bottom10 = df.sort_values("arrest_rate_per_1000", ascending=True)[
    ["ZCTA", "total_arrests", "total_pop", "arrest_rate_per_1000",
     "pct_black", "pct_hispanic", "pct_white", "pct_asian", "median_income"]
].head(10)

print("Top 10 zip codes according to arrest rate per 1000 people:")
display(top10.head())

print("Bottom 10 zip codes according to arrest rate per 1000 people:")
display(bottom10.head())

Top 10 zip codes according to arrest rate per 1000 people:


,ZCTA,total_arrests,total_pop,arrest_rate_per_1000,pct_black,pct_hispanic,pct_white,pct_asian,median_income
12,10013,2556,28215.0,90.590112,0.032430,0.077583,0.530569,0.305547,159474.0
33,10035,3173,39082.0,81.188271,0.352311,0.418121,0.140525,0.056190,40556.0
62,10451,3683,48975.0,75.201633,0.395569,0.508341,0.059990,0.011373,37111.0
50,10301,2824,39799.0,70.956557,0.227242,0.292319,0.368276,0.074876,84937.0
65,10454,2829,40368.0,70.080262,0.233725,0.702388,0.032972,0.003047,27500.0


Bottom 10 zip codes according to arrest rate per 1000 people:


,ZCTA,total_arrests,total_pop,arrest_rate_per_1000,pct_black,pct_hispanic,pct_white,pct_asian,median_income
90,11040,26,42344.0,0.614019,0.011761,0.111940,0.385509,0.438031,153511.0
87,11001,30,25714.0,1.166680,0.055145,0.154507,0.512561,0.242747,144755.0
45,10128,174,59000.0,2.949153,0.052898,0.125525,0.673881,0.103576,148705.0
43,10075,72,23865.0,3.016970,0.008213,0.053300,0.791913,0.095370,146556.0
145,11364,114,36417.0,3.130406,0.018618,0.151715,0.294286,0.502265,101222.0


Interpreting the top zip codes:

- Bronx and Harlem ZIPs (10451, 10454, 10035) are classically high-arrest areas, racially diverse, and lower-income.
- Staten Island’s 10301 shows moderate-income, diverse composition.
- 10013 (Tribeca/SoHo) is the only anomaly which is likely driven by daytime influx again (small resident base, major nightlife/commercial policing).
  - Why may this be?
    - Heavy commercial density (bars, protests, SoHo retail theft arrests, etc.) so its high rate likely reflects policing of public activity rather than residential crime (similar to what Ariel mentioned in her exploratory analysis)

Interpreting the bottom zip codes:
- These are wealthy, low-crime, majority-White/Asian neighborhoods, which should appear at the low end of arrest intensity and they do. Higher-income, majority-White ZIPs show the lowest arrest intensities; lower-income, racially diverse Bronx and Harlem ZIPs show the highest.


#### Exploratory Relationships